# QNN vs Quantum Kernel Mini Bakeoff

This notebook compares the two main routes:

- **Quantum kernel / QSVC:** usually the safer hackathon bet because the SVM training problem is convex once the kernel is computed.
- **Variational QNN:** more flexible and closer to a neural-network story, but harder to train and easier to derail with optimizer settings, initialization, and barren-plateau-like behavior.

The goal is not to make the QNN win at all costs. The goal is to decide whether it deserves stage time in your final solution.

## 1. Imports

The QNN section uses the same `SamplerQNN` pattern as the side quest and your original notebook.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score, classification_report, f1_score
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.svm import SVC

from qiskit.circuit.library import n_local, zz_feature_map
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit_algorithms.optimizers import COBYLA, SPSA
from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.neural_networks import SamplerQNN

SEED = 8398
np.random.seed(SEED)

## 2. Helpers

The dataset is quantum-tailored, but this notebook uses only four qubits by default so the QNN training loop stays manageable.

In [ ]:
def evaluate_predictions(name, y_true, y_pred, *, show_confusion=True):
    """Print metrics that are useful for balanced binary classification."""
    print(f"\n{name}")
    print("balanced accuracy:", balanced_accuracy_score(y_true, y_pred))
    print("defect F1 score  :", f1_score(y_true, y_pred, zero_division=0))
    print(classification_report(y_true, y_pred, target_names=["clean", "defect"], zero_division=0))

    if show_confusion:
        ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["clean", "defect"])
        plt.title(name)
        plt.show()


def evaluate_classifier(name, model, X_eval, y_eval, *, show_confusion=True):
    y_pred = model.predict(X_eval)
    evaluate_predictions(name, y_eval, y_pred, show_confusion=show_confusion)
    return y_pred


def balanced_subset(X_data, y_data, n_per_class=24, seed=SEED):
    """Make a small balanced set for slower quantum experiments."""
    rng = np.random.default_rng(seed)
    chosen = []
    for cls in np.unique(y_data):
        cls_idx = np.flatnonzero(y_data == cls)
        take = min(n_per_class, len(cls_idx))
        chosen.extend(rng.choice(cls_idx, size=take, replace=False).tolist())
    rng.shuffle(chosen)
    return X_data[chosen], y_data[chosen]


def latent_to_texture_images(X_latent, size=8):
    """Convert low-dimensional latent vectors into small grayscale texture images.

    The image is only a visualization/carrier for the latent features. The label is
    not created by inserting a visible blob; it is created by a quantum-kernel teacher.
    """
    grid = np.linspace(-1.0, 1.0, size)
    rr, cc = np.meshgrid(grid, grid, indexing="ij")
    images = []

    for x in X_latent:
        terms = [
            np.sin(np.pi * rr + x[0]),
            np.cos(np.pi * cc + x[1]),
            np.sin(np.pi * (rr + cc) + x[2]),
            np.cos(np.pi * (rr - cc) + x[3]),
        ]
        if len(x) > 4:
            terms.append(np.sin(np.pi * rr * cc + x[4]))
        if len(x) > 5:
            terms.append(np.cos(np.pi * (rr**2 - cc**2) + x[5]))

        texture = np.mean(terms, axis=0)
        texture = (texture - texture.min()) / (texture.max() - texture.min() + 1e-12)
        texture = 0.25 + 0.55 * texture
        images.append(texture)

    return np.array(images)


def make_quantum_teacher_dataset(
    *,
    n_samples=100,
    n_qubits=5,
    n_centers=12,
    feature_reps=1,
    image_size=8,
    seed=SEED,
):
    """Create a synthetic image dataset whose labels are generated in quantum feature space.

    A few random centers act like support vectors. The label is the sign of a weighted
    sum of quantum kernel similarities to those centers. This is an engineered task,
    not a medical claim, but it is much closer to the theory behind quantum kernels.
    """
    rng = np.random.default_rng(seed)
    X_latent = rng.uniform(0.0, 2 * np.pi, size=(n_samples, n_qubits))
    centers = rng.uniform(0.0, 2 * np.pi, size=(n_centers, n_qubits))

    feature_map = zz_feature_map(feature_dimension=n_qubits, reps=feature_reps)
    quantum_kernel = FidelityQuantumKernel(feature_map=feature_map)

    center_kernel = quantum_kernel.evaluate(x_vec=X_latent, y_vec=centers)
    weights = rng.normal(size=n_centers)
    raw_scores = center_kernel @ weights
    threshold = np.median(raw_scores)
    labels = (raw_scores > threshold).astype(int)

    images = latent_to_texture_images(X_latent, size=image_size)

    return {
        "X_latent": X_latent,
        "images": images,
        "labels": labels,
        "centers": centers,
        "weights": weights,
        "scores": raw_scores,
        "threshold": threshold,
        "feature_map": feature_map,
        "quantum_kernel": quantum_kernel,
    }


def show_labeled_textures(images, labels, scores=None, n_show=12, cols=6):
    order = np.arange(min(n_show, len(images)))
    rows = int(np.ceil(len(order) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.5, rows * 1.7))
    axes = np.ravel(axes)

    for ax, idx in zip(axes, order):
        ax.imshow(images[idx], cmap="gray_r", vmin=0, vmax=1)
        label = "DEFECT" if labels[idx] else "clean"
        suffix = "" if scores is None else f"\nscore={scores[idx]:.3f}"
        ax.set_title(f"#{idx} {label}{suffix}", fontsize=8, color="crimson" if labels[idx] else "black")
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axes[len(order):]:
        ax.axis("off")

    fig.tight_layout()
    plt.show()


def centered_kernel_alignment(K, y):
    """Centered alignment between a kernel matrix and the ideal label kernel y y^T."""
    y_signed = 2 * np.asarray(y) - 1
    Y = np.outer(y_signed, y_signed)
    n = K.shape[0]
    H = np.eye(n) - np.ones((n, n)) / n
    Kc = H @ K @ H
    Yc = H @ Y @ H
    denom = np.linalg.norm(Kc, "fro") * np.linalg.norm(Yc, "fro")
    return float(np.sum(Kc * Yc) / denom) if denom else np.nan

## 3. Generate a Smaller Quantum-Tailored Dataset

Four qubits are enough for a meaningful bakeoff and much less painful for variational training.

In [ ]:
N_SAMPLES = 80
N_QUBITS = 4
N_CENTERS = 10
FEATURE_REPS = 1
IMAGE_SIZE = 8

data = make_quantum_teacher_dataset(
    n_samples=N_SAMPLES,
    n_qubits=N_QUBITS,
    n_centers=N_CENTERS,
    feature_reps=FEATURE_REPS,
    image_size=IMAGE_SIZE,
    seed=SEED + 4,
)

X_latent = data["X_latent"]
images = data["images"]
y = data["labels"]
feature_map = data["feature_map"]
quantum_kernel = data["quantum_kernel"]

show_labeled_textures(images, y, scores=data["scores"], n_show=12, cols=6)

## 4. Split and Baselines

Start with a classical RBF SVM and a quantum-kernel SVM. If the QNN cannot beat or match these, keep it as an appendix rather than the core method.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_latent,
    y,
    test_size=0.30,
    random_state=SEED,
    stratify=y,
)

classical_rbf = Pipeline([
    ("scale", StandardScaler()),
    ("model", SVC(kernel="rbf", class_weight="balanced", random_state=SEED)),
])
classical_rbf.fit(X_train, y_train)
evaluate_classifier("Classical latent RBF SVM", classical_rbf, X_test, y_test, show_confusion=False)

X_kernel_train, y_kernel_train = balanced_subset(X_train, y_train, n_per_class=24)
qsvc = QSVC(quantum_kernel=quantum_kernel, C=1.0, class_weight="balanced")
qsvc.fit(X_kernel_train, y_kernel_train)
evaluate_classifier("Quantum Kernel QSVC", qsvc, X_test, y_test, show_confusion=False)

## 5. Build a Small `SamplerQNN`

The ansatz is deliberately shallow. If a shallow QNN works, that is useful. If it fails, increasing depth may make training harder rather than better.

In [ ]:
QNN_TRAIN_PER_CLASS = 10
QNN_REPS = 1
QNN_MAXITER = 24

X_qnn_train, y_qnn_train = balanced_subset(
    X_train,
    y_train,
    n_per_class=QNN_TRAIN_PER_CLASS,
)

qnn_feature_map = zz_feature_map(feature_dimension=N_QUBITS, reps=FEATURE_REPS)
qnn_ansatz = n_local(
    num_qubits=N_QUBITS,
    rotation_blocks=["ry", "rz"],
    entanglement_blocks="cz",
    entanglement="linear",
    reps=QNN_REPS,
    parameter_prefix="theta",
)

qnn_circuit = qnn_feature_map.compose(qnn_ansatz)


def parity_interpretation(measured_integer):
    return bin(measured_integer).count("1") % 2

sampler = StatevectorSampler(seed=SEED)
sampler_qnn = SamplerQNN(
    circuit=qnn_circuit,
    sampler=sampler,
    input_params=list(qnn_feature_map.parameters),
    weight_params=list(qnn_ansatz.parameters),
    interpret=parity_interpretation,
    output_shape=2,
)

print("QNN circuit qubits:", qnn_circuit.num_qubits)
print("trainable weights:", qnn_ansatz.num_parameters)
print("training samples:", len(X_qnn_train))

## 6. Train with COBYLA

COBYLA is a simple derivative-free default. Keep `QNN_MAXITER` small for the first run, then increase only if the model shows promise.

In [ ]:
np.random.seed(SEED)
initial_weights = np.random.uniform(-0.1, 0.1, size=qnn_ansatz.num_parameters)
optimizer = COBYLA(maxiter=max(QNN_MAXITER, qnn_ansatz.num_parameters + 2))

qnn_classifier = NeuralNetworkClassifier(
    neural_network=sampler_qnn,
    optimizer=optimizer,
    initial_point=initial_weights,
)

qnn_classifier.fit(X_qnn_train, y_qnn_train)
evaluate_classifier("SamplerQNN with COBYLA", qnn_classifier, X_test, y_test)

## 7. Optional: SPSA Optimizer

SPSA can be more noise-tolerant, especially on hardware. It is optional because it can add runtime and variance.

In [ ]:
RUN_SPSA = False

if RUN_SPSA:
    np.random.seed(SEED + 1)
    spsa_initial = np.random.uniform(-0.1, 0.1, size=qnn_ansatz.num_parameters)
    spsa_classifier = NeuralNetworkClassifier(
        neural_network=sampler_qnn,
        optimizer=SPSA(maxiter=QNN_MAXITER),
        initial_point=spsa_initial,
    )
    spsa_classifier.fit(X_qnn_train, y_qnn_train)
    evaluate_classifier("SamplerQNN with SPSA", spsa_classifier, X_test, y_test)
else:
    print("Set RUN_SPSA = True to try SPSA.")

## 8. Decision Rule for the Final Project

Use this as your team rule:

- If `QSVC` is competitive and QNN is unstable, lead with quantum kernels.
- If projected features are competitive, mention them as the hardware-friendly variant.
- If QNN wins consistently over several seeds, include it as the expressive model.
- If classical baselines win, be honest and pivot the story to dataset construction and quantum-inspired diagnostics.

## References and AI Tools Disclosure

This notebook was drafted with OpenAI Codex/GPT-5 assistance in the local hackathon workspace. The team should treat the code and results as AI-assisted prototypes and verify claims before presenting them.

AI-assisted notebooks in this `main_challenge` folder:

- `01_defect_classical_hardness_audit.ipynb`
- `02_quantum_kernel_teacher_defects.ipynb`
- `03_projected_quantum_kernel_features.ipynb`
- `04_qnn_vs_kernel_bakeoff.ipynb`
- `05_qcnn_industrial_microdefects.ipynb`
- `06_data_reuploading_qnn_microdefects.ipynb`
- `07_qnn_kernel_pivot_scoreboard.ipynb`
- `08_imbalanced_rare_defect_qnn.ipynb`
- `09_normal_only_anomaly_detection.ipynb`
- `10_heat_exchanger_network_qubo_qaoa.ipynb`
- `11_Final_QCNN_rare_defect_detection.ipynb`

References used for the quantum-ML and optimization direction:

- Cong, Choi, and Lukin, "Quantum convolutional neural networks," Nature Physics 15, 1273-1278 (2019): https://www.nature.com/articles/s41567-019-0648-8
- Perez-Salinas et al., "Data re-uploading for a universal quantum classifier," Quantum 4, 226 (2020): https://doi.org/10.22331/q-2020-02-06-226 and https://arxiv.org/abs/1907.02085
- McClean et al., "Barren plateaus in quantum neural network training landscapes," Nature Communications 9, 4812 (2018): https://www.nature.com/articles/s41467-018-07090-4
- Havlicek et al., "Supervised learning with quantum-enhanced feature spaces," Nature 567, 209-212 (2019): https://www.nature.com/articles/s41586-019-0980-2
- Scholkopf et al., "Estimating the Support of a High-Dimensional Distribution," Neural Computation 13(7), 1443-1471 (2001): https://doi.org/10.1162/089976601750264965
- Furman and Sahinidis, "Computational complexity of heat exchanger network synthesis," Computers & Chemical Engineering 25(9-10), 1371-1390 (2001): https://doi.org/10.1016/S0098-1354(01)00681-0
- Qiskit Machine Learning `SamplerQNN` documentation: https://qiskit-community.github.io/qiskit-machine-learning/stubs/qiskit_machine_learning.neural_networks.SamplerQNN.html